# 02 — The class experiment: intuition against indicators

**Reads:** `data/raw/class_experiment_results.xlsx` (the instructor's file: one row per team, the final value of
each team's two portfolios), `data/raw/prices_yahoo.csv` (SPY only, for the passive comparison).

**Writes:** `results/tables/class_experiment.csv` (the summary), `results/tables/class_experiment_teams.csv`
(one row per team, for Figure 1).

Each team of two started two $100,000 portfolios on Investopedia's simulator and held them for about two months.
Portfolio 1 was picked on intuition. Portfolio 2 was picked with indicators computed from past returns
(Sharpe ratio, Jensen's alpha, correlations, maximum-Sharpe weights) or with the class XGBoost script.

The comparison is **paired**: the same team built both portfolios, so the quantity of interest is each team's
difference, portfolio 2 minus portfolio 1. This notebook follows the instructor's script
(`reference/original_stockmarketgameresults.R`): a paired t-test and a bootstrap of the mean difference.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
TAB = ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

SEED = 6400              # course number; every random draw in this repository starts from it
B = 10_000               # bootstrap resamples
START_VALUE = 100_000

teams = pd.read_excel(RAW / "class_experiment_results.xlsx").apply(pd.to_numeric)
teams.columns = ["intuition", "indicators"]
teams["difference"] = teams["indicators"] - teams["intuition"]
teams.index.name = "team"
teams.describe().round(0)

,intuition,indicators,difference
count,45.0,45.0,45.0
mean,101179.0,103087.0,1909.0
std,5386.0,7969.0,7618.0
min,89464.0,87253.0,-13751.0
25%,97705.0,97266.0,-1985.0
50%,100928.0,102848.0,2231.0
75%,103034.0,107519.0,5623.0
max,114318.0,120787.0,22478.0


## Paired t-test and bootstrap of the mean difference

The t-test asks how many standard errors the mean difference is from zero. The bootstrap resamples the 45 team
differences with replacement 10,000 times and takes the middle 95% of the resampled means as the interval.

In [2]:
d = teams["difference"].to_numpy()
n = len(d)
mean_d, sd_d = d.mean(), d.std(ddof=1)
se_d = sd_d / np.sqrt(n)
t_stat = mean_d / se_d
p_two = 2 * stats.t.sf(abs(t_stat), n - 1)
p_one = stats.t.sf(t_stat, n - 1)                     # H1: indicators > intuition

rng = np.random.default_rng(SEED)
boot_means = rng.choice(d, size=(B, n), replace=True).mean(axis=1)
ci_lo, ci_hi = np.quantile(boot_means, [0.025, 0.975])

# how many teams it would take for a difference of this size to reach t = 1.96, at this spread
n_needed = int(np.ceil((1.96 * sd_d / mean_d) ** 2))

summary = pd.Series({
    "teams": n,
    "mean_intuition": teams["intuition"].mean(),
    "mean_indicators": teams["indicators"].mean(),
    "median_intuition": teams["intuition"].median(),
    "median_indicators": teams["indicators"].median(),
    "sd_intuition": teams["intuition"].std(ddof=1),
    "sd_indicators": teams["indicators"].std(ddof=1),
    "return_intuition_pct": 100 * (teams["intuition"].mean() / START_VALUE - 1),
    "return_indicators_pct": 100 * (teams["indicators"].mean() / START_VALUE - 1),
    "mean_difference": mean_d,
    "sd_difference": sd_d,
    "se_difference": se_d,
    "t_stat": t_stat,
    "p_two_sided": p_two,
    "p_one_sided": p_one,
    "boot_se": boot_means.std(ddof=1),
    "boot_ci95_lo": ci_lo,
    "boot_ci95_hi": ci_hi,
    "teams_indicators_ahead": int((d > 0).sum()),
    "share_intuition_above_start": (teams["intuition"] > START_VALUE).mean(),
    "share_indicators_above_start": (teams["indicators"] > START_VALUE).mean(),
    "correlation": teams["intuition"].corr(teams["indicators"]),
    "teams_needed_for_t_1_96": n_needed,
})
summary.round(4)

teams                               45.0000
mean_intuition                  101178.6076
mean_indicators                 103087.4984
median_intuition                100927.9800
median_indicators               102848.3300
sd_intuition                      5386.4045
sd_indicators                     7968.6017
return_intuition_pct                 1.1786
return_indicators_pct                3.0875
mean_difference                   1908.8909
sd_difference                     7617.6948
se_difference                     1135.5789
t_stat                               1.6810
p_two_sided                          0.0999
p_one_sided                          0.0499
boot_se                           1125.0052
boot_ci95_lo                      -339.9257
boot_ci95_hi                      4108.6822
teams_indicators_ahead              30.0000
share_intuition_above_start          0.6000
share_indicators_above_start         0.6444
correlation                          0.4017
teams_needed_for_t_1_96         

## The passive comparison

What $100,000 in SPY would have become over the same weeks. The exact start and end dates of the game are not in
the results file: **set the two dates below to the day the portfolios were opened and the day the final values
were recorded.** The number is reported in the table but nothing else depends on it.

In [3]:
GAME_START = "2026-09-01"      # confirm
GAME_END = "2026-11-06"        # confirm

spy = (pd.read_csv(RAW / "prices_yahoo.csv", parse_dates=["date"])
       .query("symbol == 'SPY'").set_index("date")["adjusted"])
spy_window = spy.loc[GAME_START:GAME_END]
summary["spy_start_date"] = spy_window.index[0].date().isoformat()
summary["spy_end_date"] = spy_window.index[-1].date().isoformat()
summary["spy_value"] = START_VALUE * spy_window.iloc[-1] / spy_window.iloc[0]
print(f"$100,000 in SPY from {summary['spy_start_date']} to {summary['spy_end_date']}: ${summary['spy_value']:,.0f}")

$100,000 in SPY from 2026-09-01 to 2026-11-06: $96,762


In [4]:
summary.rename("value").to_csv(TAB / "class_experiment.csv", header=True, index_label="statistic")
teams.round(2).to_csv(TAB / "class_experiment_teams.csv")

print(f"{n} teams. Intuition ${summary['mean_intuition']:,.0f}, indicators ${summary['mean_indicators']:,.0f}.")
print(f"Mean difference ${mean_d:,.0f} (s.d. ${sd_d:,.0f}); t = {t_stat:.2f}, "
      f"p = {p_two:.3f} two-sided, {p_one:.3f} one-sided.")
print(f"95% bootstrap interval ${ci_lo:,.0f} to ${ci_hi:,.0f}. Indicators ahead for {int((d > 0).sum())} of {n} teams.")
print(f"Teams needed for a difference of this size to reach t = 1.96: {n_needed}.")

45 teams. Intuition $101,179, indicators $103,087.
Mean difference $1,909 (s.d. $7,618); t = 1.68, p = 0.100 two-sided, 0.050 one-sided.
95% bootstrap interval $-340 to $4,109. Indicators ahead for 30 of 45 teams.
Teams needed for a difference of this size to reach t = 1.96: 62.
